In [1]:
import seaborn as sns
import pandas as pd 

df = sns.load_dataset("titanic")
df.to_csv("../data/titanic.csv", index=False)

In [2]:
df.shape

(891, 15)

In [3]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [4]:
df.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [5]:
df["survived"].value_counts(normalize=True)

survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

In [6]:
X = df.drop(columns=["survived", "alive"])
y = df["survived"]
X.head(), y.head()

(   pclass     sex   age  sibsp  parch     fare embarked  class    who  \
 0       3    male  22.0      1      0   7.2500        S  Third    man   
 1       1  female  38.0      1      0  71.2833        C  First  woman   
 2       3  female  26.0      0      0   7.9250        S  Third  woman   
 3       1  female  35.0      1      0  53.1000        S  First  woman   
 4       3    male  35.0      0      0   8.0500        S  Third    man   
 
    adult_male deck  embark_town  alone  
 0        True  NaN  Southampton  False  
 1       False    C    Cherbourg  False  
 2       False  NaN  Southampton   True  
 3       False    C  Southampton  False  
 4        True  NaN  Southampton   True  ,
 0    0
 1    1
 2    1
 3    1
 4    0
 Name: survived, dtype: int64)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# stratify가 필요한 이유는 무작위로 나누면 우연히 test에 생존자가 몰릴 수 있습니다. 그러면 train과 test의 클래스 비율이 달라져서 점수가 모델 실력이 아니라 분할 운에 좌우됩니다. 
# stratify=y는 두 쪽의 클래스 비율을 원본과 똑같이 유지합니다.
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}, {y_train.value_counts(normalize=True)}")
print(f"y_test shape: {y_test.shape}, {y_test.value_counts(normalize=True)}")


X_train shape: (712, 13)
X_test shape: (179, 13)
y_train shape: (712,), survived
0    0.616573
1    0.383427
Name: proportion, dtype: float64
y_test shape: (179,), survived
0    0.614525
1    0.385475
Name: proportion, dtype: float64


In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
dc = DummyClassifier(strategy="most_frequent")
dc.fit(X_train, y_train)
predicted = dc.predict(X_test)


In [9]:
result = {}
acc = accuracy_score(y_test, predicted)
f1 = f1_score(y_test, predicted)
result["model_name"] = "DummyClassifier"
result["accuracy"] = acc 
result["f1 score"] = f1 
print(f"accuracy: {acc}")
print(f"f1 score: {f1}")

accuracy: 0.6145251396648045
f1 score: 0.0


In [10]:
# recall = TP / (TP + FN) 
# FN = 68
# precision = TP / (TP + FP)

In [11]:
# Save to result
pd.DataFrame([result]).to_csv("../results/scores.csv", mode="a", header=False)

In [12]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

num_cols = ["age", "sibsp", "parch", "fare", "pclass"]
cat_cols = ["sex", "embarked"]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                        ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])

def evaluate(name, model):
    pipe = Pipeline([("pre", pre), ("model", model)]).fit(X_train, y_train)
    pred = pipe.predict(X_test)
    row = {"model_name": name,
           "accuracy": accuracy_score(y_test, pred),
           "f1 score": f1_score(y_test, pred),
           "auc": roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1]),
           "train_acc": pipe.score(X_train, y_train)
           }
    pd.DataFrame([row]).to_csv("../results/scores.csv", mode="a", header=False, index=False)
    return row

evaluate("LogisticRegression", LogisticRegression(max_iter=1000))

{'model_name': 'LogisticRegression',
 'accuracy': 0.8044692737430168,
 'f1 score': 0.7244094488188977,
 'auc': 0.8437417654808959,
 'train_acc': 0.8075842696629213}

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

evaluate("RandomForest", RandomForestClassifier(random_state=42))

{'model_name': 'RandomForest',
 'accuracy': 0.8044692737430168,
 'f1 score': 0.7286821705426356,
 'auc': 0.8353754940711462,
 'train_acc': 0.9831460674157303}

In [23]:
evaluate("GradientBoosting", GradientBoostingClassifier())

{'model_name': 'GradientBoosting',
 'accuracy': 0.7988826815642458,
 'f1 score': 0.7142857142857143,
 'auc': 0.8179183135704875,
 'train_acc': 0.9030898876404494}

In [24]:
evaluate("KNeighbors", KNeighborsClassifier(n_neighbors=5))

{'model_name': 'KNeighbors',
 'accuracy': 0.8156424581005587,
 'f1 score': 0.7441860465116279,
 'auc': 0.8495388669301713,
 'train_acc': 0.8497191011235955}

In [26]:
from sklearn.model_selection import cross_val_score

models = {"LogReg": LogisticRegression(max_iter=1000),
          "RF": RandomForestClassifier(max_depth=5, random_state=42),
          "GB": GradientBoostingClassifier(),
          "KN": KNeighborsClassifier(n_neighbors=5)
          }

for name, model in models.items():
    pipe = Pipeline([("pre", pre), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc")
    print(name, scores.mean(), scores.std())


LogReg 0.8556347170767233 0.023096569072263436
RF 0.8712607395797051 0.020738115696369047
GB 0.8768550578934592 0.03301176604985089
KN 0.8521162117517918 0.017777133519110606


In [27]:
from sklearn.model_selection import cross_val_score

models = {"LogReg": LogisticRegression(max_iter=1000),
          "RF": RandomForestClassifier(max_depth=5, random_state=42),
          "GB": GradientBoostingClassifier(max_depth=2, random_state=42),
          "KN": KNeighborsClassifier(n_neighbors=5)
          }

for name, model in models.items():
    pipe = Pipeline([("pre", pre), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc")
    print(name, scores.mean(), scores.std())


LogReg 0.8556347170767233 0.023096569072263436
RF 0.8712607395797051 0.020738115696369047
GB 0.881818841497525 0.02323930210271035
KN 0.8521162117517918 0.017777133519110606


In [29]:
evaluate("RF_depths5", RandomForestClassifier(max_depth=5, random_state=42))

{'model_name': 'RF_depths5',
 'accuracy': 0.8100558659217877,
 'f1 score': 0.7166666666666667,
 'auc': 0.8470355731225298,
 'train_acc': 0.8595505617977528}

In [30]:
evaluate("GB_depths2", GradientBoostingClassifier(max_depth=2, random_state=42))

{'model_name': 'GB_depths2',
 'accuracy': 0.7932960893854749,
 'f1 score': 0.6890756302521008,
 'auc': 0.8193017127799737,
 'train_acc': 0.8609550561797753}

In [33]:
X_train["family_size"] = X_train["sibsp"] + X_train["parch"] + 1
X_test["family_size"] = X_test["sibsp"] + X_test["parch"] + 1
num_cols = ["age", "sibsp", "parch", "fare", "pclass", "family_size"]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                        ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])

def evaluate(name, model):
    pipe = Pipeline([("pre", pre), ("model", model)]).fit(X_train, y_train)
    pred = pipe.predict(X_test)
    row = {"model_name": name,
           "accuracy": accuracy_score(y_test, pred),
           "f1 score": f1_score(y_test, pred),
           "auc": roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1]),
           "train_acc": pipe.score(X_train, y_train)
           }
    pd.DataFrame([row]).to_csv("../results/scores.csv", mode="a", header=False, index=False)
    return row

evaluate("RF_depths5_family", RandomForestClassifier(max_depth=5, random_state=42))


{'model_name': 'RF_depths5_family',
 'accuracy': 0.8156424581005587,
 'f1 score': 0.736,
 'auc': 0.8467061923583663,
 'train_acc': 0.8441011235955056}

In [34]:
evaluate("GB_depths2_family", GradientBoostingClassifier(max_depth=2, random_state=42))

{'model_name': 'GB_depths2_family',
 'accuracy': 0.8044692737430168,
 'f1 score': 0.7154471544715447,
 'auc': 0.8198945981554677,
 'train_acc': 0.8693820224719101}

In [35]:

models = {"LogReg": LogisticRegression(max_iter=1000),
          "RF": RandomForestClassifier(max_depth=5, random_state=42),
          "GB": GradientBoostingClassifier(max_depth=2, random_state=42),
          "KN": KNeighborsClassifier(n_neighbors=5)
          }

for name, model in models.items():
    pipe = Pipeline([("pre", pre), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc")
    print(name, scores.mean(), scores.std())

LogReg 0.8555938697318007 0.023259716511825956
RF 0.8688599157721415 0.01834574508394772
GB 0.8832189845160064 0.02501087400189353
KN 0.8479764204215614 0.024288337305285533
